# Getting started: runtime-sky one-arm and Gen-2 Michelson TDI

This notebook shows the **new staged implementation**, where the static state is independent of sky position and `theta`, `phi` are supplied at runtime.

Two paths are demonstrated:

1. **One-arm path:** waveform → six directed links → select one link.
2. **Full TDI path:** waveform → six directed links → prepared 42-job operator → `X, Y, Z`.

The notebook intentionally separates static, waveform and runtime work so it is clear what can be reused in parameter estimation.

## 0. Imports and numerical settings

Adjust the package import path to match your checkout. The settings below are representative rather than universally optimal; `row_chunk_size` and `lag_block_size` should be benchmarked on the target machine.

In [1]:
import numpy as np

from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
    apply_one_arm_wdm_runtime_from_context,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMStaticPlan
from WDM_LISAresponse.code.lisa_response.TDI_with_one_arm_plan import (
    precompute_gen2_michelson_wdm_static,
    precompute_gen2_michelson_wdm_waveform_runtime,
    gen2_michelson_wdm_from_static_waveform,
)

Nf = 100
L_trunc = 25
q = None
d = 4
A_frac = 0.25
calc_m0 = False

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 16,
    "tl_tp_interp_kind": "linear",
    "row_chunk_size": 2048,
    "lag_block_size": 17,
    "batch_chunk": 32,
}


## 1. Provide the common inputs

The following objects must already exist:

- `times`: uniformly sampled reception-time grid, shape `(N,)`, seconds;
- `waveform_plus`, `waveform_cross`: time-domain polarizations, shape `(N,)`;
- `orbits`: orbit provider exposing `compute_position`;
- `theta`, `phi`: runtime source angles in radians.

The assertions below make the expected contract explicit.

In [2]:
from few.waveform import GenerateEMRIWaveform
from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)


# Orbit model
orbit_data = read_orbit_file("esa-trailing-orbits.h5")
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)

orbits = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)


# FEW generator: return h_plus and h_cross separately
td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs={
        "pad_output": True,
        "odd_len": True,
    },
    return_list=True,
    force_backend="cpu",
)

In [3]:
# Time and sky parameters
dt = 10.0
T_obs = 1.0
t0 = float(orbit_data.t0 + 365.25 * 24.0 * 3600.0)

theta = np.pi / 2
phi = 0.0


times, waveform_plus, waveform_cross, padding_info = (
    generate_windowed_padded_waveform(
        waveform_model="emri",
        t0=t0,
        dt=dt,
        T_obs=T_obs,
        model_kwargs={
            "td_gen": td_gen,
            "m1": 1.0e6,
            "m2": 10.0,
            "a": 0.9,
            "p0": 8.5,
            "e0": 0.3,
            "x0": 1.0,
            "dist": 1.0,
            "theta": theta,
            "phi": phi,
            "qK": np.pi / 3,
            "phiK": np.pi / 3,
            "Phi_phi0": np.pi / 3,
            "Phi_theta0": 0.0,
            "Phi_r0": np.pi / 3,
            "eps": 1.0e-2,
        },
        window_function="tukey",
        window_kwargs={
            "alpha": 0.005,
            "tukey_factor": 4.0,
        },
        wdm_Nf=None,
        return_padding_info=True,
    )
)


assert times.ndim == 1
assert waveform_plus.shape == times.shape
assert waveform_cross.shape == times.shape
assert np.allclose(np.diff(times), times[1] - times[0])

# Path A: one-arm response only

This path avoids constructing recursive TDI delays and the 42-job TDI plan. It is the cleanest route for one-arm validation, mismatch studies and response diagnostics.

## A0. Build one-arm static state

This is reusable across **all waveforms and sky positions** on the same time grid, WDM basis and orbit model.

In [4]:
one_arm_grid = _build_wdm_dT_grid(
    times,
    Nf=Nf,
    d=d,
    A_frac=A_frac,
    q=q,
    calc_m0=calc_m0,
)

one_arm_static = OneArmWDMStaticPlan.build_from_context(
    one_arm_grid,
    orbits,
    L_trunc=L_trunc,
    variable_shift_kwargs=SHIFT_KWARGS,
)

print("WDM shape:", one_arm_static.wdm.Nt, one_arm_static.wdm.Nf)
print("Directed links:", one_arm_static.links)
print("Persistent kernel memory [MiB]:", one_arm_static.kernel_memory_bytes / 2**20)


WDM shape: 32190 100
Directed links: ((2, 1), (3, 2), (1, 3), (3, 1), (2, 3), (1, 2))
Persistent kernel memory [MiB]: 1.6015625


## A1. Transform one waveform

Only the two DWTs are waveform-dependent here. Set `parallel_wdm_transforms=True` to run the independent plus and cross transforms on two Python threads.

In [5]:
one_arm_wave = _build_wdm_dT_context_from_wdm(
    times,
    waveform_plus,
    waveform_cross,
    one_arm_grid["wdm"],
    parallel_wdm_transforms=True,
)

print("w_plus shape:", one_arm_wave["w_t_plus"].shape)
print("w_cross shape:", one_arm_wave["w_t_cross"].shape)


w_plus shape: (32190, 100)
w_cross shape: (32190, 100)


## B. Evaluate a runtime sky position

This computes all six directed links in one batch. Select a single link afterwards; for example `(2, 1)` means sender spacecraft 2 and receiver spacecraft 1.

In [6]:
wdm, link_result = apply_one_arm_wdm_runtime_from_context(
    one_arm_wave,
    one_arm_static,
    theta,
    phi,
    return_profile=True,
)

link_w, one_arm_profile = link_result
w_21 = link_w[(2, 1)]

print("Available links:", tuple(link_w))
print("Selected link shape:", w_21.shape)
one_arm_profile


Available links: ((2, 1), (3, 2), (1, 3), (3, 1), (2, 3), (1, 2))
Selected link shape: (32190, 100)


{'n_jobs': 6,
 'runtime_interpolation_s': 0.2887594169988006,
 'assembly_s': 2.6165909160008596,
 'total_s': 2.905499584001518,
 'kernel_context_build_s': 0.4968027920003806,
 'interpolation_table_memory_bytes': 1679360,
 'assembly_backend': 'production',
 'assembly_precision': 'complex64',
 'returned_on_device': False}

# Path B: full Gen-2 Michelson TDI

This path builds the same one-arm static machinery **plus** the recursive orbit-dependent delay libraries and the prepared 42-job Michelson operator.

## A0. Build the complete static state

Reuse this object across waveforms and runtime sky locations.

In [7]:
static = precompute_gen2_michelson_wdm_static(
    times,
    orbits,
    Nf=Nf,
    L_trunc=L_trunc,
    q=q,
    d=d,
    A_frac=A_frac,
    calc_m0=calc_m0,
    variable_shift_kwargs=SHIFT_KWARGS,
)

print("Static keys:", sorted(static.keys()))
print("TDI jobs:", static["tdi_plan"].shift_plan.num_jobs)
print("TDI kernel memory [MiB]:", static["tdi_plan_kernel_memory_bytes"] / 2**20)


Static keys: ['A_frac', 'L_trunc', 'Nf', 'calc_m0', 'd', 'job_templates', 'kind', 'one_arm_interpolation_table_memory_bytes', 'one_arm_shift_kwargs', 'one_arm_static_build_s', 'one_arm_static_plan', 'orbits', 'q', 'shifts_X', 'shifts_Y', 'shifts_Z', 'tdi_batch_shift_kwargs', 'tdi_delay_build_s', 'tdi_delays_from_cache', 'tdi_plan', 'tdi_plan_build_s', 'tdi_plan_kernel_memory_bytes', 'times', 'times_p', 'use_jax', 'wdm', 'wdm_grid_build_s', 'wdm_grid_context']
TDI jobs: 42
TDI kernel memory [MiB]: 1052.1084594726562


## A1. Transform one waveform

In [8]:
waveform_state = precompute_gen2_michelson_wdm_waveform_runtime(
    static,
    waveform_plus,
    waveform_cross,
    parallel_wdm_transforms=True,
)

waveform_state["waveform_context_build_s"]


1.853187374999834

## B. Evaluate one sky and return `X, Y, Z`

This call computes the six sky-dependent links and immediately applies the prepared TDI plan.

In [9]:
(X, Y, Z), profile = gen2_michelson_wdm_from_static_waveform(
    static,
    waveform_state,
    theta,
    phi,
    return_profile=True,
)

print("X, Y, Z shapes:", X.shape, Y.shape, Z.shape)
profile


X, Y, Z shapes: (32190, 100) (32190, 100) (32190, 100)


{'theta': 1.5707963267948966,
 'phi': 0.0,
 'one_arm': {'n_jobs': 6,
  'runtime_interpolation_s': 0.531570457998896,
  'assembly_s': 3.0948358750029,
  'total_s': 3.626492875002441,
  'kernel_context_build_s': 0.31540770799983875,
  'interpolation_table_memory_bytes': 1679360,
  'assembly_backend': 'production',
  'assembly_precision': 'complex64',
  'returned_on_device': False},
 'tdi': {'n_sources': 6,
  'n_jobs': 42,
  'n_outputs': 3,
  'batch_chunk': 32,
  'assembly_backend': 'production',
  'assembly_precision': 'complex64',
  'total_s': 21.84819133400015,
  'plan_build_s': 33.13284358300007,
  'device_plan_cached': True,
  'device_plan_memory_bytes': 1108623804,
  'full_materialized_output_bytes': 540792000,
  'returned_output_bytes': 38628000,
  'host_transfer_bytes_saved': 502164000,
  'tdi_execution_mode': 'indexed_accumulate',
  'tdi_plan_build_s': 33.278439666999475,
  'tdi_kernel_memory_bytes': 1103215680},
 'tdi_apply_s': 21.87589008299983,
 'total_runtime_s': 26.427855375

## Optional: inspect the intermediate six links before TDI

Use this when comparing one-arm and full-TDI costs or validating a particular directed link.

In [10]:
wdm, runtime_result = apply_one_arm_wdm_runtime_from_context(
    waveform_state["wave_context"],
    static["one_arm_static_plan"],
    theta,
    phi,
    return_profile=True,
)

link_w, link_profile = runtime_result
X2, Y2, Z2 = static["tdi_plan"].apply(link_w)

np.testing.assert_allclose(X2, X)
np.testing.assert_allclose(Y2, Y)
np.testing.assert_allclose(Z2, Z)

link_profile


{'n_jobs': 6,
 'runtime_interpolation_s': 0.26426829200136126,
 'assembly_s': 2.8427306250014226,
 'total_s': 3.1070457080022607,
 'kernel_context_build_s': 0.31540770799983875,
 'interpolation_table_memory_bytes': 1679360,
 'assembly_backend': 'production',
 'assembly_precision': 'complex64',
 'returned_on_device': False}

# What changes at each inference level?

| Change | Recompute |
|---|---|
| Time grid, WDM settings, orbit model, shift configuration | Static state |
| Intrinsic waveform parameters | Two DWTs and subsequent response |
| `theta`, `phi` only | Six one-arm links and TDI application |
| Need only a link, not TDI | Stop after `apply_one_arm_wdm_runtime_from_context` |

## Main performance controls

- **Approximation and cost:** `Nf`, `q`, `L_trunc`, `tl_tp_mode`, interpolation points and precision.
- **JAX scheduling:** `row_chunk_size`, `lag_block_size`, `batch_chunk`.
- **Waveform scheduling:** `parallel_wdm_transforms`.
- The runtime-sky `OneArmWDMStaticPlan` currently evaluates all six links in one materialized batch; fixed-sky `OneArmWDMPlan` additionally exposes parallel link groups.